> **Disclaimer:** This notebook is for *educational and research training purposes only*. It does not constitute medical, clinical, or diagnostic advice. All computational predictions are approximations from machine learning models and must not be treated as experimental results. See the full [DISCLAIMER](https://github.com/JobAiReady/lagos-bio-design/blob/main/DISCLAIMER.md) and [PRIVACY POLICY](https://github.com/JobAiReady/lagos-bio-design/blob/main/PRIVACY_POLICY.md).
>
> **Attribution:** This notebook adapts the [VibeGen Colab Demo](https://github.com/lamm-mit/ModeShapeDiffusionDesign/tree/main/colab_demo) by Bo Ni & Markus J. Buehler (MIT LAMM), licensed under [Apache-2.0](https://github.com/lamm-mit/ModeShapeDiffusionDesign/blob/main/LICENSE.txt). Model weights from [lamm-mit/VibeGen](https://huggingface.co/lamm-mit/VibeGen) on HuggingFace.

# Module 3 Bonus: Dynamics-First Protein Design with VibeGen
## Lab: From Vibrational Fingerprints to Novel Proteins

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Generate a de novo protein conditioned on a target **vibrational mode** (pattern of motion) using VibeGen's agentic dual-model architecture — then fold it and visualize the result.

### Prerequisites
- Completed Module 3 (Binder Design)
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)
- ~20 minutes for model download (~16 GB of checkpoints)

### Deliverable
A comparison of the VibeGen-designed protein's predicted dynamics vs. the target vibrational mode, plus a 3D visualization of the folded structure.

> ⚠️ **GPU Required & Large Download.** Go to Runtime → Change runtime type → T4 GPU. The model weights are ~16 GB total — ensure you have sufficient disk space in your Colab runtime.

---
## Background: Why Dynamics?

In Modules 2 and 3, you designed proteins using a **structure-first** pipeline: RFDiffusion generates a static backbone shape, ProteinMPNN finds a sequence for it, and AlphaFold validates the fold. This pipeline answers one question: *does the protein fold correctly?*

But proteins aren't rigid statues — they're **dynamic molecular machines**. Their biological functions (enzyme catalysis, signal transduction, mechanical response) depend on how they vibrate, flex, and change shape. A protein that folds correctly but moves wrong is like a car with a perfect body but a broken engine.

**VibeGen** (Ni & Buehler, MIT, 2026) inverts the design paradigm: instead of specifying a target *shape*, you specify a target *motion* — a **normal mode** describing how the protein should collectively vibrate. The model generates amino acid sequences that produce proteins exhibiting those exact dynamics.

### The Agentic Design Loop

VibeGen uses two collaborating AI models in an iterative loop:

```
Target vibrational mode
        ↓
  ProteinDesigner  →  generates 20 candidate sequences
        ↓
  ProteinPredictor  →  evaluates dynamic accuracy of each
        ↓
  Best candidates selected (lowest prediction error)
        ↓
  Fold with OmegaFold  →  visualize structure + dynamics
```

This is an **agentic** architecture: propose → critique → select. The designer hallucinates *motions* (not shapes), and the predictor acts as quality control. The generated proteins are **de novo** — they share no significant similarity with natural proteins, expanding the design space beyond what evolution has produced.

### Structure-First vs. Dynamics-First

| | Structure-First (Modules 2-3) | Dynamics-First (VibeGen) |
|---|---|---|
| **Input** | Target backbone shape | Target vibrational mode |
| **Generator** | RFDiffusion | ProteinDesigner (language diffusion) |
| **Validator** | AlphaFold (RMSD < 2 Å) | ProteinPredictor (mode amplitude error) |
| **Output** | Sequence that folds correctly | Sequence that moves correctly |
| **Design space** | Shapes nature hasn't made | Motions nature hasn't made |

### Key Terms

| Term | Definition |
|------|-----------|
| **Normal Mode** | A pattern of collective atomic motion (vibration) in a protein. Low-frequency modes describe large-scale functional movements: domain opening, hinge bending, breathing motions |
| **VibeGen** | Generative AI framework (MIT, 2026) that designs de novo proteins conditioned on target vibrational modes using language diffusion models |
| **Agentic Design Loop** | Iterative propose→critique→refine architecture: ProteinDesigner generates candidates, ProteinPredictor evaluates dynamic accuracy |
| **Language Diffusion Model** | A diffusion model that operates on discrete token sequences (like amino acids) rather than continuous 3D coordinates. Same family as image diffusion but applied to biological "language" |
| **OmegaFold** | A single-sequence protein structure prediction model. Used here to fold VibeGen-designed sequences without requiring multiple sequence alignments |
| **Mode Amplitude** | The magnitude of displacement at each residue position for a given vibrational mode. The design target that VibeGen optimizes for |

---
## Setup

This section installs all dependencies. The first run takes ~15-20 minutes due to model downloads.

In [ ]:
# Step 0.1: Verify GPU and install OmegaFold + DSSP
import torch
assert torch.cuda.is_available(), 'GPU not found! Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

import os, sys

# Install OmegaFold (for folding designed sequences)
from huggingface_hub import hf_hub_download, snapshot_download

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

dssp_file = '/opt/bin/mkdssp'
if not os.path.exists(dssp_file):
    print('Installing OmegaFold...')
    !pip install -q git+https://github.com/Bo-Ni/OmegaFold_0.git

    print('Installing DSSP...')
    hf_hub_download(
        repo_id="lamm-mit/ProteinMechanicsDiffusionDesign",
        filename="Model_files/mkdssp",
        local_dir='/content',
        local_dir_use_symlinks=False,
        revision='main'
    )
    os.popen(f"mv /content/Model_files/mkdssp {dssp_file}").read()
    !chmod u+x /opt/bin/mkdssp
    print('Done.')
else:
    print('OmegaFold & DSSP already installed.')

In [ ]:
# Step 0.2: Install additional Python packages
!pip install -q biopython kornia einops einops-exts pytorch-warmup ema-pytorch accelerate py3Dmol fair-esm torchinfo beartype==0.18.5

In [ ]:
# Step 0.3: Clone VibeGen source code from GitHub
import json, time, os, sys, glob

code_dir = '/content/ModeShapeDiffusionDesign/'
if not os.path.isdir(code_dir):
    os.system('git clone -q https://github.com/lamm-mit/ModeShapeDiffusionDesign.git')
sys.path.append(code_dir)

# Verify the import works
import VibeGen.UtilityPack as UPack
import VibeGen.DataSetPack as DPack
import VibeGen.ModelPack as MPack
import VibeGen.TrainerPack as TPack
import VibeGen.JointSamplingPack as SPack
print('VibeGen packages imported successfully.')

In [ ]:
# Step 0.4: Download pretrained model weights from HuggingFace (~16 GB)
# This includes ProteinDesigner + ProteinPredictor checkpoints and test data
trained_duo_dir = '/content/trained_duo/'

if os.path.exists(trained_duo_dir):
    print('Model weights already downloaded.')
else:
    print('Downloading VibeGen model weights from HuggingFace...')
    print('This may take 15-20 minutes on first run.')
    UPack.create_path(trained_duo_dir)
    snapshot_download(
        repo_id="lamm-mit/VibeGen",
        local_dir=trained_duo_dir,
        repo_type="model"
    )
    print('Done! Model weights downloaded.')

---
## Step 1: Initialize the Agentic Duo

Load both models that form VibeGen's agentic loop:
- **ProteinDesigner** (~448M parameters): generates candidate sequences from a target vibrational mode
- **ProteinPredictor** (~448M parameters): evaluates how well each candidate matches the target dynamics

Together they have ~896M parameters — comparable to a medium-sized language model.

In [ ]:
# Initialize ProteinDesigner + ProteinPredictor
import importlib
import pickle
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import shutil
from einops import rearrange

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')
torch.cuda.empty_cache()

# === Control Keys ===
CKeys = dict(
    Working_Mode='Sampling',
    n_try_w_PD=20,       # Designer generates 20 candidates per input
    n_keep_w_PP=2,       # Predictor picks best 2 and worst 2
    PD_cond_scal=1.0,
    PP_cond_scal=1.0,
    Debug=1,
    wk_dir='/content/wk_dir/',
    trained_duo_dir='/content/trained_duo/',
)

CKeys['Debug_Datakey'] = 1
CKeys['Debug_Sampling'] = 1
CKeys['SilentRun'] = 0

if os.path.exists(CKeys['wk_dir']):
    print('Working directory exists.')
else:
    UPack.create_path(CKeys['wk_dir'])

# === ProteinDesigner config ===
PD_CKeys = dict(
    Working_Mode=3, IF_FirstRun=1, Resume_from_where='LAST',
    Problem_ID=1, Debug=1, SilentRun=0,
    Debug_DataPack=0, Debug_ModelPack=0, Debug_TrainlPack=0,
    testset_raio=0.10
)

# === ProteinPredictor config ===
PP_CKeys = dict(
    Working_Mode=3, IF_FirstRun=2, Resume_from_where='LAST',
    Problem_ID=2, Debug=1, SilentRun=0,
    Debug_DataPack=1, Debug_ModelPack=1, Debug_TrainlPack=1,
    testset_raio=0.10
)

# === Data and model paths ===
PD_PKeys = dict(
    seq_len=128, maxdata=10000000, batch_size=256, testset_ratio=0.10,
    text_seq_len=128, num_train_epochs=4000, gradient_accumulation_steps=8,
    pk_data_pack=CKeys['trained_duo_dir']+'/ProteinDesigner/data_pack.pickle',
    pk_model_pack=CKeys['trained_duo_dir']+'/ProteinDesigner/model_pack.pickle',
)
PP_PKeys = dict(
    seq_len=128, maxdata=10000000, batch_size=256, testset_ratio=0.10,
    text_seq_len=128, num_train_epochs=4000, gradient_accumulation_steps=8,
    pk_data_pack=CKeys['trained_duo_dir']+'/ProteinPredictor/data_pack.pickle',
    pk_model_pack=CKeys['trained_duo_dir']+'/ProteinPredictor/model_pack.pickle',
)

# === Load data packs ===
PD_DataKeys = dict(
    maxdata=10000000, batch_size=256, testset_ratio=0.1,
    min_AA_seq_len=0, max_AA_seq_len=128, max_used_Seg_Num=1,
    X_Key=['Mode7_NormDisAmp'], X_mode_num=1,
    **{'ESM-2_Model': 'esm2_t30_150M_UR50D'},
    image_channels=33, Xnormfac=[9.513197412136874],
    ynormfac=1.0, tokenizer_X=None, tokenizer_y=None
)
PD_df_test = pd.read_pickle(CKeys['trained_duo_dir']+'/ProteinDesigner/df_test.pk')

PP_DataKeys = dict(
    maxdata=10000000, batch_size=256, testset_ratio=0.1,
    min_AA_seq_len=0, max_AA_seq_len=128, max_used_Seg_Num=1,
    X_Key=['Mode7_NormDisAmp'], X_mode_num=1,
    **{'ESM-2_Model': 'esm2_t30_150M_UR50D'},
    image_channels=33, Xnormfac=[9.513197412136874],
    ynormfac=1.0, tokenizer_X=None, tokenizer_y=None
)

print(f'Test set loaded: {len(PD_df_test)} proteins with known vibrational modes.')
print(f'Input feature: {PD_DataKeys["X_Key"][0]} (Normal Mode 7 displacement amplitudes)')

In [ ]:
# Build ProteinDesigner model architecture and load weights
PD_ModelKeys = {
    'text_con_embed_dim': 768, 'text_pos_embed_dim': 32,
    'text_seq_len': 128, 'UNet_cond_images_channels': 1,
    'UNet_channels': 33, 'UNet_dim': 256,
    'UNet_lowres_cond': False, 'UNet_self_cond': True,
    'UNet_text_embed_dim': 800, 'EImg_seq_size': 128,
    'EImg_cond_drop_prob': 0.1, 'EImg_sample_step': 96,
    'text_embed_AsInPut_dim': 1
}

# Build UNet backbone for ProteinDesigner
PD_wk_unet = MPack.Unet_OneD(
    channels=PD_ModelKeys['UNet_channels'],
    lowres_cond=PD_ModelKeys['UNet_lowres_cond'],
    self_cond=PD_ModelKeys['UNet_self_cond'],
    cond_images_channels=PD_ModelKeys['UNet_cond_images_channels'],
    init_cross_embed=True, init_cross_embed_kernel_sizes=(3, 7, 15),
    init_conv_kernel_size=7,
    dim=PD_ModelKeys['UNet_dim'], dim_mults=(1, 2, 4, 8),
    learned_sinu_pos_emb_dim=16, num_time_tokens=2,
    text_embed_dim=PD_ModelKeys['UNet_text_embed_dim'],
    cond_on_text=True, attn_pool_text=True,
    attn_dim_head=64, attn_heads=8, attn_pool_num_latents=32,
    max_text_len=PD_ModelKeys['text_seq_len'],
    num_resnet_blocks=1,
    layer_attns=(False, True, True, True), layer_attns_depth=1,
    layer_cross_attns=(False, True, True, True),
    use_linear_attn=False, use_linear_cross_attn=False,
    cross_embed_downsample=True, cross_embed_downsample_kernel_sizes=(2, 4),
    memory_efficient=False, use_global_context_attn=True,
    scale_skip_connection=True, pixel_shuffle_upsample=True,
    ff_mult=2., layer_mid_attns_depth=1, attend_at_middle=True,
    combine_upsample_fmaps=True, init_conv_to_final_conv_residual=False,
    final_resnet_block=True, final_conv_kernel_size=3, resize_mode='nearest',
    layer_attns_add_text_cond=True, dropout=0.,
    CKeys={'Debug_Level': 0},
).to(device)

# Wrap in ProteinDesigner diffusion model
PD_wk_ProteinDesigner = MPack.ProteinDesigner_Base(
    PD_wk_unet, elucidated=True,
    seq_obj_max_size=PD_ModelKeys['EImg_seq_size'],
    text_embed_input_dim=PD_ModelKeys['text_embed_AsInPut_dim'],
    text_con_embed_dim=PD_ModelKeys['text_con_embed_dim'],
    text_pos_embed_dim=PD_ModelKeys['text_pos_embed_dim'],
    cond_drop_prob=PD_ModelKeys['EImg_cond_drop_prob'],
    timestep=PD_ModelKeys['EImg_sample_step'],
    proteinLanguageModel=PD_DataKeys['ESM-2_Model'],
    CKeys={'Debug_Level': 0},
).to(device)

# Load pretrained weights
PD_ckpt_name = CKeys['trained_duo_dir'] + '/ProteinDesigner/Last_ckpt.pt'
PD_checkpoint = torch.load(PD_ckpt_name, map_location=device)
PD_state_dict = PD_checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(PD_state_dict.items()):
    if k.startswith(unwanted_prefix):
        PD_state_dict[k[len(unwanted_prefix):]] = PD_state_dict.pop(k)
PD_wk_ProteinDesigner.load_state_dict(PD_state_dict)

PD_ck_info = {
    'completed_updating_steps': PD_checkpoint['completed_updating_steps'],
    'step_num': PD_checkpoint['step_num'],
}
del PD_state_dict, PD_checkpoint, PD_wk_unet
print(f'ProteinDesigner loaded. Training steps: {PD_ck_info["completed_updating_steps"]}')

In [ ]:
# Build ProteinPredictor model architecture and load weights
PP_ModelKeys = {
    'text_con_embed_dim': 640, 'text_pos_embed_dim': 0,
    'text_seq_len': 128, 'UNet_cond_images_channels': 33,
    'UNet_channels': 1, 'UNet_dim': 256,
    'UNet_lowres_cond': False, 'UNet_self_cond': True,
    'UNet_text_embed_dim': 640, 'EImg_seq_size': 128,
    'EImg_cond_drop_prob': 0.1, 'EImg_sample_step': 96,
    'text_embed_AsInPut_dim': 0
}

PP_wk_unet = MPack.Unet_OneD(
    channels=PP_ModelKeys['UNet_channels'],
    lowres_cond=PP_ModelKeys['UNet_lowres_cond'],
    self_cond=PP_ModelKeys['UNet_self_cond'],
    cond_images_channels=PP_ModelKeys['UNet_cond_images_channels'],
    init_cross_embed=True, init_cross_embed_kernel_sizes=(3, 7, 15),
    init_conv_kernel_size=7,
    dim=PP_ModelKeys['UNet_dim'], dim_mults=(1, 2, 4, 8),
    learned_sinu_pos_emb_dim=16, num_time_tokens=2,
    text_embed_dim=PP_ModelKeys['UNet_text_embed_dim'],
    cond_on_text=True, attn_pool_text=True,
    attn_dim_head=64, attn_heads=8, attn_pool_num_latents=32,
    max_text_len=PP_ModelKeys['text_seq_len'],
    num_resnet_blocks=1,
    layer_attns=(False, True, True, True), layer_attns_depth=1,
    layer_cross_attns=(False, True, True, True),
    use_linear_attn=False, use_linear_cross_attn=False,
    cross_embed_downsample=True, cross_embed_downsample_kernel_sizes=(2, 4),
    memory_efficient=False, use_global_context_attn=True,
    scale_skip_connection=True, pixel_shuffle_upsample=True,
    ff_mult=2., layer_mid_attns_depth=1, attend_at_middle=True,
    combine_upsample_fmaps=True, init_conv_to_final_conv_residual=False,
    final_resnet_block=True, final_conv_kernel_size=3, resize_mode='nearest',
    layer_attns_add_text_cond=True, dropout=0.,
    CKeys={'Debug_Level': 0},
).to(device)

PP_wk_ProteinPredictor = MPack.ProteinPredictor_Base(
    PP_wk_unet, elucidated=True,
    seq_obj_max_size=PP_ModelKeys['EImg_seq_size'],
    text_embed_input_dim=PP_ModelKeys['text_embed_AsInPut_dim'],
    text_con_embed_dim=PP_ModelKeys['text_con_embed_dim'],
    text_pos_embed_dim=PP_ModelKeys['text_pos_embed_dim'],
    cond_drop_prob=PP_ModelKeys['EImg_cond_drop_prob'],
    timestep=PP_ModelKeys['EImg_sample_step'],
    proteinLanguageModel=PP_DataKeys['ESM-2_Model'],
    CKeys={'Debug_Level': 0},
).to(device)

PP_ckpt_name = CKeys['trained_duo_dir'] + '/ProteinPredictor/Last_ckpt.pt'
PP_checkpoint = torch.load(PP_ckpt_name, map_location=device)
PP_state_dict = PP_checkpoint['model']
for k, v in list(PP_state_dict.items()):
    if k.startswith('_orig_mod.'):
        PP_state_dict[k[len('_orig_mod.'):]] = PP_state_dict.pop(k)
PP_wk_ProteinPredictor.load_state_dict(PP_state_dict)

PP_ck_info = {
    'completed_updating_steps': PP_checkpoint['completed_updating_steps'],
    'step_num': PP_checkpoint['step_num'],
}
del PP_state_dict, PP_checkpoint, PP_wk_unet
print(f'ProteinPredictor loaded. Training steps: {PP_ck_info["completed_updating_steps"]}')
print(f'\nBoth models ready! The agentic duo is armed.')

---
## Step 2: Select a Target Vibrational Mode

We'll pick a real protein's vibrational mode from the test set as our design target. The model was trained on **Normal Mode 7** displacement amplitudes — a low-frequency collective motion that captures functional dynamics.

**What's happening:** We're selecting the "vibrational fingerprint" we want our designed protein to reproduce. This is like giving RFDiffusion a target shape, except here we're giving VibeGen a target *motion*.

In [ ]:
# Pick a test protein's vibrational mode as our target
test_idx = 0  # Change this to explore different targets
test_protein = PD_df_test.iloc[test_idx]

# Extract the target normal mode amplitude profile
target_mode = test_protein['Mode7_NormDisAmp']
original_seq = test_protein['Seq']
seq_len = len(original_seq)

print(f'Target protein #{test_idx}:')
print(f'  Original sequence length: {seq_len} residues')
print(f'  Original sequence: {original_seq[:50]}...')
print(f'  Normal mode amplitudes: {len(target_mode)} values')

# Visualize the target vibrational mode
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(range(len(target_mode[:seq_len])), target_mode[:seq_len], 'b-', linewidth=1.5)
ax.fill_between(range(len(target_mode[:seq_len])), target_mode[:seq_len], alpha=0.3)
ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Mode 7 Displacement Amplitude', fontsize=12)
ax.set_title('Target Vibrational Fingerprint (Normal Mode 7)', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'\nThis is the "motion blueprint" — VibeGen will design a new protein that vibrates like this.')

---
## Step 3: Generate Candidates with the Agentic Duo

Now the two models collaborate:
1. **ProteinDesigner** generates 20 candidate sequences conditioned on the target mode
2. **ProteinPredictor** evaluates each candidate's predicted dynamics
3. The best 2 and worst 2 are selected for comparison
4. Best candidates are folded with OmegaFold

**What's happening:** This is the agentic loop in action — propose → critique → select. The designer doesn't know if its outputs are good; the predictor provides the quality signal.

In [ ]:
# Run the agentic design loop
# ProteinDesigner proposes 20 candidates → ProteinPredictor picks best/worst 2
SPack.Joint_Sampling(
    CKeys=CKeys,
    PD_CKeys=PD_CKeys,
    PP_CKeys=PP_CKeys,
    PD_PKeys=PD_PKeys,
    PP_PKeys=PP_PKeys,
    PD_DataKeys=PD_DataKeys,
    PP_DataKeys=PP_DataKeys,
    PD_ModelKeys=PD_ModelKeys,
    PP_ModelKeys=PP_ModelKeys,
    PD_wk_ProteinDesigner=PD_wk_ProteinDesigner,
    PP_wk_ProteinPredictor=PP_wk_ProteinPredictor,
    PD_ck_info=PD_ck_info,
    PP_ck_info=PP_ck_info,
    df_test=PD_df_test,
    device=device,
)

---
## Step 4: Examine the Results

The joint sampling saves detailed results including:
- Designed sequences and their predicted dynamics
- Comparison plots (target vs. predicted vibrational modes)
- Folded structures (PDB files from OmegaFold)
- Secondary structure analysis (DSSP)

Let's examine what the agentic duo produced.

In [ ]:
# Find the output directory
wk_dir = CKeys['wk_dir']
result_dirs = sorted(glob.glob(os.path.join(wk_dir, 'PD_*')))

if result_dirs:
    result_dir = result_dirs[-1]
    print(f'Results directory: {result_dir}')

    # List generated files
    all_files = glob.glob(os.path.join(result_dir, '**', '*'), recursive=True)
    pdb_files = [f for f in all_files if f.endswith('.pdb')]
    csv_files = [f for f in all_files if f.endswith('.csv')]
    png_files = [f for f in all_files if f.endswith('.png')]

    print(f'\nGenerated files:')
    print(f'  PDB structures: {len(pdb_files)}')
    print(f'  CSV data files: {len(csv_files)}')
    print(f'  PNG plots: {len(png_files)}')

    # Show the recording CSV
    reco_files = [f for f in csv_files if 'reco' in f]
    if reco_files:
        df_reco = pd.read_csv(reco_files[0])
        print(f'\nDesign summary:')
        display(df_reco)
else:
    print('No results found. Please run Step 3 first.')

In [ ]:
# Visualize the best designed protein in 3D (if PDB files were generated)
import py3Dmol

if pdb_files:
    # Use the first (best) PDB
    best_pdb = sorted(pdb_files)[0]
    print(f'Visualizing: {os.path.basename(best_pdb)}')

    with open(best_pdb, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {
        'colorscheme': {'prop': 'b', 'gradient': 'rwb', 'min': 50, 'max': 90}
    }})
    view.zoomTo()
    view.show()

    print('\nColor scale: Red (low confidence) → White → Blue (high confidence)')
    print('This protein was designed to match the target vibrational fingerprint.')
    print('Its sequence is entirely de novo — no natural protein has this exact sequence.')
else:
    print('No PDB files found. OmegaFold may still be running or encountered an error.')

---
## Summary

In this bonus lab you:

1. **Specified a target motion** — selected a vibrational mode (Normal Mode 7) as the design blueprint
2. **Ran the agentic duo** — ProteinDesigner proposed 20 candidate sequences, ProteinPredictor evaluated their dynamic accuracy
3. **Folded the best candidates** — OmegaFold predicted 3D structures for the top designs
4. **Visualized the result** — a de novo protein designed not for a specific shape, but for a specific pattern of motion

### How This Differs from the Structure-First Pipeline

| Step | Structure-First (Module 2-3) | Dynamics-First (This Lab) |
|------|-----|------|
| Design input | 3D backbone coordinates | Vibrational mode amplitudes |
| Design model | RFDiffusion (denoises 3D coords) | VibeGen Designer (denoises sequence tokens) |
| Quality check | AlphaFold RMSD < 2 Å | VibeGen Predictor mode error |
| What's new | A shape | A motion |

### Why This Matters

- **Adaptive therapeutics**: proteins that flex to match different drug targets
- **Dynamic biomaterials**: self-healing scaffolds, impact-resistant fibers
- **The Lassa connection**: The GPC target in Module 4 undergoes conformational changes during viral entry. Dynamics-aware binders could exploit those motions.

### References

- Ni, B. & Buehler, M.J. "VibeGen: Agentic end-to-end de novo protein design for tailored dynamics using a language diffusion model." *Matter* (2026). [DOI](https://www.sciencedirect.com/science/article/pii/S259023852600069X)
- [GitHub: lamm-mit/ModeShapeDiffusionDesign](https://github.com/lamm-mit/ModeShapeDiffusionDesign) (Apache-2.0)
- [HuggingFace: lamm-mit/VibeGen](https://huggingface.co/lamm-mit/VibeGen)

**Next:** Module 4 — Applying the design pipeline to Lassa Fever (Capstone Project)